In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 1) Download & extract UEC-Food100
!wget http://foodcam.mobi/dataset100.zip \
     -O /content/drive/MyDrive/UEC-Food100.zip
!unzip /content/drive/MyDrive/UEC-Food100.zip \
       -d /content/drive/MyDrive/UEC-Food100/

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
import torch.optim as optim
from torchvision.transforms import InterpolationMode
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from tqdm import tqdm
from PIL import Image

from torch.optim.lr_scheduler import CosineAnnealingLR
import copy
from sklearn.metrics import top_k_accuracy_score

In [ ]:
# Custom Dataset with Bounding Boxes
class UECFood100WithBBox(ImageFolder):
    def __init__(self, root, transform=None):
        super().__init__(root, transform=transform)
        self.bbox_cache = {}

    def parse_bb_file(self, class_dir):
        bb_file = os.path.join(class_dir, 'bb_info.txt')
        bbox_dict = {}
        if os.path.exists(bb_file):
            with open(bb_file, 'r') as f:
                lines = f.readlines()[1:]
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        img_id, x1, y1, x2, y2 = parts
                        bbox_dict[f"{img_id}.jpg"] = (int(x1), int(y1), int(x2), int(y2))
        return bbox_dict

    def __getitem__(self, index):
        path, target = self.samples[index]
        class_dir = os.path.dirname(path)

        if class_dir not in self.bbox_cache:
            self.bbox_cache[class_dir] = self.parse_bb_file(class_dir)

        bbox_dict = self.bbox_cache[class_dir]
        img_name = os.path.basename(path)

        img = Image.open(path).convert('RGB')

        if img_name in bbox_dict:
            x1, y1, x2, y2 = bbox_dict[img_name]
            img = img.crop((x1, y1, x2, y2))

        if self.transform is not None:
            img = self.transform(img)

        return img, target

In [ ]:
# --- Paths and config ---
data_dir    = '/content/drive/MyDrive/UEC-Food100/UECFOOD100'
num_classes = 100
device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Transforms ---
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.5, 1.0), interpolation=InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# --- Dataset & Dataloaders ---
full_ds = UECFood100WithBBox(root=data_dir, transform=None)
train_size = int(0.8 * len(full_ds))
test_size  = len(full_ds) - train_size
train_ds, test_ds = random_split(full_ds, [train_size, test_size])

# assign transforms
train_ds.dataset.transform = train_transform
test_ds.dataset.transform  = test_transform

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=4)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=4)

In [ ]:
# --- Model: EfficientNet-B0 ---
weights = EfficientNet_B0_Weights.IMAGENET1K_V1
model   = efficientnet_b0(weights=weights)
model   = model.to(device)

# --- Modify the head to match 100 classes ---
in_f = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(in_f, 256),
    nn.ReLU(),
    nn.BatchNorm1d(256),
    nn.Dropout(p=0.2),
    nn.Linear(256, num_classes)
).to(device)

In [ ]:
# --- Define loss, optimizer and scheduler ---
criterion = nn.CrossEntropyLoss(label_smoothing=0.15)
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=18)  # total_epochs - warmup_epochs

In [ ]:
# --- EMA Teacher Initialization ---
teacher_model = copy.deepcopy(model)
teacher_model.eval()
for p in teacher_model.parameters():
    p.requires_grad = False

def update_ema_teacher(student_model, teacher_model, decay=0.999):
    for teacher_param, student_param in zip(teacher_model.parameters(), student_model.parameters()):
        teacher_param.data.mul_(decay).add_(student_param.data, alpha=1 - decay)

In [ ]:
# --- Training loop ---
def train_one_epoch(model, teacher_model, dataloader, criterion, optimizer, alpha=0.7, T=4.0):
    model.train()
    running_loss, correct, total = 0, 0, 0
    kl_loss = nn.KLDivLoss(reduction="batchmean")

    loop = tqdm(dataloader, desc="Training", leave=False)
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        student_outputs = model(images)
        with torch.no_grad():
            teacher_outputs = teacher_model(images)

        loss_ce = criterion(student_outputs, labels)
        soft_student = nn.functional.log_softmax(student_outputs / T, dim=1)
        soft_teacher = nn.functional.softmax(teacher_outputs / T, dim=1)
        loss_kd = kl_loss(soft_student, soft_teacher) * (T ** 2)

        loss = alpha * loss_kd + (1 - alpha) * loss_ce
        loss.backward()
        optimizer.step()

        # EMA update
        update_ema_teacher(model, teacher_model)

        running_loss += loss.item() * images.size(0)
        preds = student_outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        loop.set_postfix(loss=loss.item(), acc=100. * correct / total)

    return running_loss / total, correct / total

In [ ]:
# --- Evaluate the loss, accuracy ---
def evaluate(model, dataloader, criterion):
    model.eval()
    running_loss, correct, total = 0, 0, 0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        loop = tqdm(dataloader, desc="Evaluating", leave=False)
        for images, labels in loop:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            preds = outputs.argmax(dim=1).cpu()
            all_probs.extend(probs)
            all_labels.extend(labels.cpu().numpy())

            correct += (preds == labels.cpu()).sum().item()
            total += labels.size(0)
            loop.set_postfix(loss=loss.item(), acc=100. * correct / total)

    top1_acc = correct / total
    top5_acc = top_k_accuracy_score(all_labels, all_probs, k=5)

    return running_loss / total, top1_acc, top5_acc

In [ ]:
# --- Full training ---
def train_model(model, teacher_model, train_loader, test_loader, criterion, optimizer, scheduler, epochs=20, warmup_epochs=2):
    for epoch in range(epochs):
        if epoch < 5:
          for name, param in model.named_parameters():
            if 'classifier' in name:
              param.requires_grad = True
            else:
              param.requires_grad = False
        else:
          for name, param in model.named_parameters():
            param.requires_grad = False
            if 'features.6' in name or 'features.7' in name or 'classifier' in name:
                param.requires_grad = True

        print(f"\n===== Epoch {epoch + 1}/{epochs} =====")
        if epoch < warmup_epochs:
            lr_scale = (epoch + 1) / warmup_epochs
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr_scale * 5e-4
            print(f"Warmup LR: {optimizer.param_groups[0]['lr']:.6f}")
        else:
            scheduler.step()
            print(f"LR after scheduler: {optimizer.param_groups[0]['lr']:.6f}")

        train_loss, train_top1 = train_one_epoch(model, teacher_model, train_loader, criterion, optimizer, alpha=0.7, T=4.0)
        test_loss, test_top1, test_top5 = evaluate(model, test_loader, criterion)

        print(f"Train Loss: {train_loss:.4f}, Top-1 Acc: {train_top1 * 100:.2f}%")
        print(f"Test  Loss: {test_loss:.4f}, Top-1 Acc: {test_top1 * 100:.2f}%, Top-5 Acc: {test_top5 * 100:.2f}%")


In [ ]:
def test_model(model, test_loader, criterion):
    print("\n===== Final Evaluation on Test Set =====")
    test_loss, test_top1, test_top5 = evaluate(model, test_loader, criterion)
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Top-1 Accuracy: {test_top1 * 100:.2f}%")
    print(f"Top-5 Accuracy: {test_top5 * 100:.2f}%")

In [ ]:
# --- Run training ---
train_model(model, teacher_model, train_loader, test_loader, criterion, optimizer, scheduler, epochs=20)


===== Epoch 1/20 =====
Warmup LR: 0.000250


Train Loss: 1.2961, Top-1 Acc: 26.38%
Test  Loss: 3.3106, Top-1 Acc: 47.37%, Top-5 Acc: 76.44%

===== Epoch 2/20 =====
Warmup LR: 0.000500


Training:   0%|          | 0/359 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Train Loss: 1.1209, Top-1 Acc: 48.21%
Test  Loss: 2.9901, Top-1 Acc: 57.01%, Top-5 Acc: 82.84%

===== Epoch 3/20 =====
LR after scheduler: 0.000496


Training:   0%|          | 0/359 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Train Loss: 1.0475, Top-1 Acc: 56.96%
Test  Loss: 2.8360, Top-1 Acc: 60.70%, Top-5 Acc: 85.62%

===== Epoch 4/20 =====
LR after scheduler: 0.000485


... (30 outputs left out)

Training:   0%|          | 0/359 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Train Loss: 0.6094, Top-1 Acc: 99.03%
Test  Loss: 1.8931, Top-1 Acc: 83.43%, Top-5 Acc: 95.89%

===== Epoch 20/20 =====
LR after scheduler: 0.000000


Training:   0%|          | 0/359 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
                                                                                

Train Loss: 0.6089, Top-1 Acc: 99.12%
Test  Loss: 1.8747, Top-1 Acc: 83.75%, Top-5 Acc: 96.10%


In [ ]:
# --- Test the performance on test set ---
test_model(model, test_loader, criterion)


===== Final Evaluation on Test Set =====


Evaluating:   0%|          | 0/90 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
                                                                                

Test Loss: 1.8747
Top-1 Accuracy: 83.75%
Top-5 Accuracy: 96.10%
